In [2]:
import pandas as pd

## 1. Comienzo por filtrar el dataset:

### 1.1 Primero saco las muestras de paises non-US:

In [ ]:
chunks = pd.read_json("goodreads_books.json", lines=True, chunksize=100_000)

us_list = []
for chunk in chunks:
    us_list.append(chunk[chunk["country_code"] == "US"])

df_us = pd.concat(us_list, ignore_index=True)
print(df_us.shape)
#Guardarlo en un json!!!

(2360165, 29)


### 1.2 Ahora me tengo que quedar nomas con los libros que son en ingles:

Consideramos los libros en ingles los que tienen "languade_code" = ["en", "eng", "en-US", "en-GB", "en-CA", "enm", "ang", "sco", "gem"]

Las ultimas 4 opciones son ingles, pero antiguos (afectara en el rating del libro?... Acaso como a las personas de hoy en dia les cosatara mas atenderlo y por eso le daran bajo rating?...)

## 2. Reorganizo valores de features:

### 2.1 Comienzo por reagrupar los valores de "format"

In [ ]:
def normalize_format(value):
    if not isinstance(value, str):
        return "UNKNOWN"
    s = value.strip().lower()
    if s == "":
        return "UNKNOWN"

    # Ebook / digital
    ebook_terms = [
        "ebook", "e-book", "kindle", "nook", "ibook", "epub", "pdf",
        "online fiction", "online free", "online read", "online", "digital",
        "e book", "e-book", "ebook and paperback", "ebook, paperback",
        "nook ebook", "e-book .pdf", "e book, paperback", "e-pub", "ebook pdf"
    ]

    # Audio formats
    audio_terms = [
        "audio", "audiobook", "audio cd", "mp3", "cassette", "audible",
        "digital audio", "audio lecture", "audio book", "audiobook, audible",
        "audio cassette", "mp3 audio", "mp3 book"
    ]

    # Hardcover / hardback
    hardcover_terms = [
        "hardcover", "hard cover", "hardback", "hardbound", "slipcased",
        "hardcover first edition", "hardcover comic", "hardcover (large print)",
        "faux leather", "hardcover with slipcase", "hardcover art"
    ]

    # Paperback / softcover
    paperback_terms = [
        "paperback", "softcover", "soft cover", "trade paperback",
        "mass market paperback", "paper back", "paperbound", "pocket paperback",
        "capa mole", "paper", "brochura", "broche", "book", "brochure",
        "brossura", "paperback with dvd", "paperback with flaps",
        "paperback (large print)", "perfect paperback", "paperback manga"
    ]

    # Comic / manga / graphic
    comic_terms = [
        "comic", "graphic novel", "manga", "webcomic", "fancomic", "comic book",
        "digital comic", "comic book", "graphic"
    ]

    s_norm = s.replace("&amp;", "&")

    if any(term in s_norm for term in ebook_terms) and not any(term in s_norm for term in audio_terms):
        return "EBOOK"
    if any(term in s_norm for term in audio_terms):
        return "AUDIO"
    if any(term in s_norm for term in hardcover_terms):
        return "HARDCOVER"
    if any(term in s_norm for term in paperback_terms):
        return "PAPERBACK"
    if any(term in s_norm for term in comic_terms):
        return "COMIC"

    return "OTHER"

In [ ]:
#Le aplico la funcion para reagrupar:
if "format_group" not in df_us.columns:
    df_us["format_group"] = df_us["format"].apply(normalize_format)

print(df_us["format_group"].value_counts())